In [11]:
import itertools
#正则化图自回归流交通数据预测模
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os
import numpy as np
import pandas as pd
from scipy.stats import entropy
import math
from dataclasses import dataclass
from typing import Optional
from tqdm.auto import tqdm

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


确定数据集

In [19]:
import os

# 1. 设定当前要处理的数据集名称 
dataset_name = "metr-la" 
fname="METRLA"
dataset="metr-la"
# 2. 自动生成输入输出路径
data_path = f"{dataset_name}/{fname}.npz"
excel_out_path = f"{dataset_name}/{fname}_period_features.xlsx"
flows_path = f"{dataset_name}/{fname}_flows.npz"
speed_path = f"{dataset_name}/{fname}_speed.npz"
# datafile = np.load(data_path)
# print(datafile['data'].shape)
print(f"🚦 当前激活数据集: {dataset_name}")
print(f"📥 输入数据路径: {data_path}")


# 确保目标文件夹存在，防止保存时报错
os.makedirs(dataset_name, exist_ok=True)


🚦 当前激活数据集: metr-la
📥 输入数据路径: metr-la/METRLA.npz


核心算法

In [20]:
import os
import numpy as np
import pandas as pd
from scipy.stats import entropy
import scipy.signal
import math
from concurrent.futures import ThreadPoolExecutor # ✅ 关键修改1：引入线程池
from tqdm import tqdm                

# 每步采样间隔（分钟），用于小时<->步数换算
STEPS_PER_HOUR = 12

class TrafficPeriodExtractor:
    """
    交通流量周期特征提取器。
    核心功能：从给定的交通节点流量时序数据中，利用物理误差与信息熵联合驱动的启发式寻找机制，自动搜索并挑选出每个节点最显著的 3 个物理周期（Short / Middle / Long）。
    这些提取到的标量周期特征参数，后续可供下游的时空图网络（如 GFlow）利用。
    """
    def __init__(self, window_size=12, min_period_h=1, max_period_h=168):
        self.w = window_size
        self.min_T_steps = min_period_h * STEPS_PER_HOUR
        self.max_T_steps = max_period_h * STEPS_PER_HOUR
        self.h = STEPS_PER_HOUR

    def _impute_missing_values(self, data):
        """
        【新增功能】：数据清洗与缺失值/0值补全
        利用 Pandas 的线性插值，平滑填补传感器丢失或损坏时产生的 0 和 NaN。
        """
        s = pd.Series(data)
        # 1. 将 0 值或负数异常值替换为 NaN，统一作为缺失值处理
        # （注：如果是流量数据，深夜极少数真实的0也会被周围的值平滑，对宏观周期无负面影响；对于速度数据，0 绝对是异常停机）
        s = s.replace(0.0, np.nan)
        s[s < 0] = np.nan 
        # 2. 线性插值补全内部的缺失片段
        s = s.interpolate(method='linear')
        
    def precompute_rolling_sum(self, data):
        """
        数据预处理：使用滑动窗口计算累加和，以平滑原始时序数据。
        """
        cumsum = np.cumsum(np.insert(data, 0, 0))
        return cumsum[self.w:] - cumsum[:-self.w]

    def extract_differences_for_T(self, rolling_sum, len_data, T):
        """
        计算差异：给定某个候选周期 T 后，将平滑后的原序列和平移 T 后的序列做绝对值相减。
        """
        valid_len = len_data - self.w - T
        diffs = np.zeros(len_data)
        if valid_len > 0:
            diffs[:valid_len] = np.abs(rolling_sum[:valid_len] - rolling_sum[T:T+valid_len])
        return diffs

    def filter_by_threshold(self, diffs, T_steps):
        """
        寻峰与错位判定：通过 scipy.signal.find_peaks 寻找差异序列的波峰，
        从而判定周期错位幅度，过滤掉低于自适应阈值的较小波动。
        """
        valid_diffs = diffs[diffs > 0]
        if len(valid_diffs) == 0:
            return []
            
        threshold = np.mean(valid_diffs) + 1.5 * np.std(valid_diffs)
        
        # 寻峰距离与当前周期保持温和的弱相关，防止极值点分裂
        dynamic_distance = max(self.w, int(T_steps * 0.3))
        
        peaks, _ = scipy.signal.find_peaks(diffs, distance=dynamic_distance)
        if len(peaks) > 0:
            valid_peaks = peaks[diffs[peaks] > threshold]
            return valid_peaks.tolist()
        return []

    def calculate_E_curr(self, valid_peaks, T_steps, mean_diff, mean_data):
        """
        【修改】复合正则化信息目标函数
        E_curr = 结构信息熵(Entropy) + 全局对齐误差(Alignment Loss) + 尺度惩罚(Scale Penalty)
        """
        if len(valid_peaks) < 2:
            return float('inf')
            
        J = np.diff(valid_peaks)
        max_interval = np.max(J)
        bins = np.arange(0, max_interval + self.h * 2, self.h)
        
        hist, _ = np.histogram(J, bins=bins)
        prob = (hist + 1e-6) / np.sum(hist + 1e-6)
        
        # 1. 结构规律项
        entropy_val = entropy(prob, base=2)
        
        # 2. 全局对齐误差项
        error_ratio = mean_diff / (mean_data + 1e-6)
        lambda_1 = 10.0  
        alignment_loss = lambda_1 * error_ratio
        
        # 3. 尺度/复杂度惩罚项：【改为：长度惩罚】
        # 周期时长越长，带来的复杂度提升越大，惩罚分值线性或对数级增高。
        # 目的是强力压制 120h/168h 这种“因为跨度太大而包含了几个小周期”的虚假长跨度，
        # 强制算法优先承认、提取 12h/24h 的底层原生短周期波动。
        T_hours = T_steps / self.h
        lambda_2 = 0.5  # 基础惩罚基数，可根据期望的压制力度微调
        # 这里采用对数增长模型：周期越长，惩罚越重，但不至于让真正的长周期完全丧失竞争力
        scale_penalty = lambda_2 * math.log(T_hours + 1.0)
        
        return entropy_val + alignment_loss + scale_penalty


    def simulated_annealing_search_T(self, data, max_iter=200):
        """
        核心启发式搜索算法：利用模拟退火 (Simulated Annealing) 在给定范围（如 6h 到 168h）内搜索 T。
        相比于暴力穷举，此方法在广阔连续域寻找最佳周期有着极高的效率，并能避免陷入局部最优。
        """
        rolling_sum = self.precompute_rolling_sum(data)
        len_data = len(data)
        
        # 预先计算序列本身的均值，用于后续的归一化误差评估
        mean_data = np.mean(rolling_sum)
        
        rng = np.random.default_rng()
        T_curr = rng.integers(self.min_T_steps, self.max_T_steps + 1)
        
        diffs_curr = self.extract_differences_for_T(rolling_sum, len_data, T_curr)
        peaks_curr = self.filter_by_threshold(diffs_curr, T_curr)
        mean_diff_curr = np.mean(diffs_curr)
        # 将误差均值与数据均值传入目标函数
        E_curr = self.calculate_E_curr(peaks_curr, T_curr, mean_diff_curr, mean_data)
        
        temp = 10.0  
        alpha = 0.95 
        explored_history = {T_curr: E_curr}
        
        for i in range(max_iter):
            if rng.random() < 0.2:
                T_new = rng.integers(self.min_T_steps, self.max_T_steps + 1)
            else:
                step_hours = rng.integers(-6, 7)  
                if step_hours == 0: step_hours = 1
                step = step_hours * STEPS_PER_HOUR
                T_new = T_curr + step
            
            T_new = max(self.min_T_steps, min(T_new, self.max_T_steps))
            
            if T_new not in explored_history:
                diffs_new = self.extract_differences_for_T(rolling_sum, len_data, T_new)
                peaks_new = self.filter_by_threshold(diffs_new, T_new)
                mean_diff_new = np.mean(diffs_new)
                E_new = self.calculate_E_curr(peaks_new, T_new, mean_diff_new, mean_data)
                explored_history[T_new] = E_new
            else:
                E_new = explored_history[T_new]
                
            delta = E_new - E_curr
            if delta < 0 or rng.random() < math.exp(-delta / (temp + 1e-9)):
                T_curr = T_new
                E_curr = E_new
                
            temp *= alpha
            
        return explored_history

    def extract(self, data):
        """
        特征去重与提取：在模拟退火搜索出的历史中寻找代价值最小的几个候选周期 T。
        使用 margin 规则过滤掉过于相近的周期，最终输出 3 个差异度显著的“短、中、长 (Short, Middle, Long)”周期特征。
        """
        # 【新增】：在任何特征提取开始之前，先对数据进行缺失值和0值的补全平滑
        cleaned_data = self._impute_missing_values(data)
        explored_history = self.simulated_annealing_search_T(data, max_iter=200)
        valid_history = {T: E for T, E in explored_history.items() if E != float('inf')}
        
        if not valid_history:
            return []
            
        sorted_by_E = sorted(valid_history.items(), key=lambda item: item[1])
        margin = 12 * STEPS_PER_HOUR
        top_3_T = []
        
        for T_candidate_steps, E_val in sorted_by_E:
            too_close = False
            for selected_T_steps, _ in top_3_T:
                if abs(T_candidate_steps - selected_T_steps) < margin:
                    too_close = True
                    break
            if not too_close:
                top_3_T.append((T_candidate_steps, E_val))
                if len(top_3_T) == 3:
                    break
                    
        # 按照特征周期的时间跨度从小到大排序 (Short, Middle, Long)
        top_3_T.sort(key=lambda item: item[0])
        
        results = []
        for T_steps, E_val in top_3_T:
            T_hours_rounded = round(T_steps / STEPS_PER_HOUR, 1)
            T_steps_rounded = round(float(T_steps), 1)
            results.append((T_steps_rounded, T_hours_rounded, E_val))
            
        return results

def process_single_node(args):
    """
    单节点处理函数：提取单节点的 Top-3 周期参数并整理成行数据格式。
    """
    node_idx, node_flow, extractor_params = args
    extractor = TrafficPeriodExtractor(**extractor_params)
    top_3 = extractor.extract(node_flow)
    
    row_data = {"Node_ID": node_idx}
    labels = ["Short", "Middle", "Long"]
    
    for i, (T_steps, T_hours, E_val) in enumerate(top_3):
        label = labels[i] if i < 3 else f"Rank{i+1}"
        row_data[f"{label}_T_hours"] = T_hours
        row_data[f"{label}_T_steps"] = T_steps
        row_data[f"{label}_E_curr"] = round(E_val, 4)
        
    return row_data



In [21]:
import os
import numpy as np

if os.path.exists(data_path):
    print(f"正在加载数据表 {data_path}")
    loaded_npz = np.load(data_path, allow_pickle=True)
    keys = list(loaded_npz.keys())

    # 1) 常见优先级：arr_0 / data / x
    if "arr_0" in loaded_npz:
        data = loaded_npz["arr_0"]
    elif "data" in loaded_npz:
        data = loaded_npz["data"]
    elif "x" in loaded_npz:  # 训练样本文件常见键
        data = loaded_npz["x"]
    # 2) METR-LA 的 pandas 打包格式
    elif "df/block0_values" in loaded_npz:
        data = loaded_npz["df/block0_values"]  # shape: (T, N)
   
    # 统一转为 flow_data: (T, N)
    if data.ndim == 2:
        time_steps, num_nodes = data.shape
        flow_data = data
    elif data.ndim == 3:
        # (T, N, C) -> 取第0通道
        time_steps, num_nodes, _ = data.shape
        flow_data = data[:, :, 0]
    elif data.ndim == 4:
        # (B, T, N, C) -> 展平批次后取第0通道
        b, t, n, c = data.shape
        data = data.reshape(b * t, n, c)
        time_steps, num_nodes, _ = data.shape
        flow_data = data[:, :, 0]
    else:
        raise ValueError(f"不支持的数据维度: {data.ndim}, keys={keys}")

    print(f"数据读取成功, 序列长度 = {time_steps} 步, 节点数 = {num_nodes}, keys = {keys}")
    extractor_params = {
        "window_size": 24, 
        "min_period_h": 24,   
        "max_period_h": 168
    }
    
    print("\n【开始全节点线程池批量特征提取】")
    tasks = [(node_idx, flow_data[:, node_idx], extractor_params) for node_idx in range(num_nodes)]
    results_list = []

   
    with ThreadPoolExecutor() as executor:
        for result in tqdm(executor.map(process_single_node, tasks), total=num_nodes, desc="Processing Nodes"):
            if result:
                results_list.append(result)
        
    df_results = pd.DataFrame(results_list)
    expected_cols = ["Node_ID", 
                     "Short_T_hours", "Short_T_steps", "Short_E_curr",
                     "Middle_T_hours", "Middle_T_steps", "Middle_E_curr",
                     "Long_T_hours", "Long_T_steps", "Long_E_curr"]
    final_cols = [c for c in expected_cols if c in df_results.columns]
    df_results = df_results[final_cols]
    
    # 将结果保存到 Excel（会自动覆盖同名文件，使用动态路径）
    df_results.to_excel(excel_out_path, index=False)
    
    print(f"🎉 成功! 所有特征已导出至 Excel: {excel_out_path}")
    print("--------------------------------------------------")
else:
    print(f"警告：找不到数据集 {data_path}，当前需有此文件以执行。")

正在加载数据表 metr-la/METRLA.npz
数据读取成功, 序列长度 = 34272 步, 节点数 = 207, keys = ['data']

【开始全节点线程池批量特征提取】


Processing Nodes: 100%|██████████| 207/207 [00:05<00:00, 34.72it/s]

🎉 成功! 所有特征已导出至 Excel: metr-la/METRLA_period_features.xlsx
--------------------------------------------------


特征拼接

PEMS04和08数据集的 3 个通道分别是 [流量 (Flow), 占有率 (Occupy), 速度 (Speed)]  03和07数据集只有一个流量的特征

In [26]:

# 强制 3D 格式 (T, N, C)，兼容常见 npz 与 metr-la
npz_data = np.load(data_path, allow_pickle=True)

if "arr_0" in npz_data:
    original_data = npz_data["arr_0"]
elif "data" in npz_data:
    original_data = npz_data["data"]
elif "df/block0_values" in npz_data:   # metr-la 常见键
    original_data = npz_data["df/block0_values"]  # shape: (T, N)
# 统一成 (T, N, C)
original_3d = original_data[:, :, np.newaxis] if original_data.ndim == 2 else original_data

# 按数据集选择用于构建相位特征的基础通道
# 按数据集选择用于构建相位特征的基础通道
match dataset_name:
    case "PEMS08":
        flow_data = original_3d[:, :, 0]
        speed_data = original_3d[:, :, 2]
    case "PEMS04":
        flow_data = original_3d[:, :, 0]
        speed_data = original_3d[:, :, 2]
    case "PEMS03":
        flow_data = original_3d[:, :, 0]
    case "pemsbay":
        speed_data = original_3d[:, :, 0]  
    case "PEMS07":
        flow_data = original_3d[:, :, 0]
    case "metr-la":
        speed_data = original_3d[:, :, 0]   # METR-LA 常见为单通道 speed
    case _:
        raise ValueError(f"不支持的数据集: {dataset_name}")


time_steps, num_nodes = flow_data.shape

df_periods = pd.read_excel(excel_out_path)
if "Node_ID" in df_periods.columns:
    df_periods = df_periods.sort_values("Node_ID").reset_index(drop=True)

features = np.zeros((time_steps, num_nodes, 3), dtype=np.float32)
indices = np.arange(time_steps)
period_cols = ['Short_T_steps', 'Middle_T_steps', 'Long_T_steps']

for node_idx in tqdm(range(num_nodes), desc=f"Building Phases ({dataset_name})"):
    node_flow = flow_data[:, node_idx]
    periods = df_periods.loc[node_idx, period_cols].values

    for p_idx, p_samples in enumerate(periods):
        if pd.isna(p_samples) or p_samples <= 0:
            continue

        # 根据周期步长划分前后相位
        phase_mask = ((indices % p_samples) / p_samples) >= 0.5

        # 计算两个相位的均值填充
        features[~phase_mask, node_idx, p_idx] = np.mean(node_flow[~phase_mask]) if np.any(~phase_mask) else 0.0
        features[phase_mask, node_idx, p_idx] = np.mean(node_flow[phase_mask]) if np.any(phase_mask) else 0.0

# 拼接并保存
# 拼接并保存
match dataset_name:
    case "PEMS08" | "PEMS04":  
        flow_out = np.concatenate([flow_data[:, :, np.newaxis], features], axis=-1)
        speed_out = np.concatenate([speed_data[:, :, np.newaxis], features], axis=-1)
        np.savez_compressed(flows_path, arr_0=flow_out)
        np.savez_compressed(speed_path, arr_0=speed_out)
        print(f"🎉 拼接成功！Flow文件: {flows_path} | Speed文件: {speed_path}")
        print(f" 最终 Shape: {flow_out.shape} (通道0: 原数据 | 附带通道: 短/中/长周期)")
    case "PEMS03" | "PEMS07" :
        flow_out = np.concatenate([flow_data[:, :, np.newaxis], features], axis=-1)
        np.savez_compressed(flows_path, arr_0=flow_out)
        print(f"🎉 拼接成功！文件已存至: {flows_path}")
        print(f" 最终 Shape: {flow_out.shape} (通道0: 原数据 | 附带通道: 短/中/长周期)")
    case "metr-la"| "PEMSBAY":
        speed_out = np.concatenate([speed_data[:, :, np.newaxis], features], axis=-1)
        np.savez_compressed(speed_path, arr_0=speed_out)
        print(f"🎉 拼接成功！文件已存至: {speed_path}")
        print(f" 最终 Shape: {speed_out.shape} (通道0: 原数据 | 附带通道: 短/中/长周期)")   
    case _:
        raise ValueError(f"不支持的数据集: {dataset_name}")



Building Phases (metr-la): 100%|██████████| 207/207 [00:00<00:00, 680.58it/s]


🎉 拼接成功！文件已存至: metr-la/METRLA_speed.npz
 最终 Shape: (34272, 207, 4) (通道0: 原数据 | 附带通道: 短/中/长周期)


In [27]:

def seq2seq_io_data(
    df, time_steps
):
    """
    # x: (epoch_size, input_length, num_nodes, input_dim)
    # y: (epoch_size, output_length, num_nodes, output_dim)
    """

    num_samples, num_nodes = df.shape[0], df.shape[1]
    data = df if df.ndim == 3 else np.expand_dims(df, axis=-1)
    x, y = [], []
    x_offsets = np.arange(-(12 - 1), 1, 1)
    y_offsets = np.sort(np.arange(1, time_steps + 1, 1))
    min_t = abs(min(x_offsets))
    max_t = abs(num_samples - abs(max(y_offsets)))  # Exclusive
    for t in range(min_t, max_t,2):
        x_t = data[t + x_offsets, ...]
        y_t = data[t + y_offsets, ...]
        x.append(x_t)
        y.append(y_t)
    x = np.stack(x, axis=0)
    y = np.stack(y, axis=0)
    return x, y


In [28]:
import numpy as np
import os
def generate_train_val_test(dataset,nfile,time_steps ):
    filename=dataset+"/"+nfile
    data=np.load(filename)
    df= data['arr_0']
    
    x, y = seq2seq_io_data( df,time_steps  )
    num_samples = x.shape[0]
    num_test = round(num_samples * 0.2)
    num_train = round(num_samples * 0.8)
    num_val = num_samples - num_test - num_train

    # train
    x_train, y_train = x[:num_train], y[:num_train]
    # val
    x_val, y_val = (
        x[num_train: num_train + num_val],
        y[num_train: num_train + num_val],
    )
    # test
    x_test, y_test = x[-num_test:], y[-num_test:]

    for cat in ["train", "val", "test"]:
        _x, _y = locals()["x_" + cat], locals()["y_" + cat]
        print(cat, "x: ", _x.shape, "y:", _y.shape)
        np.savez_compressed(
            os.path.join(dataset, str(cat)+"_"+str(time_steps)+".npz"),
            x=_x,
            y=_y,
        )

In [29]:
generate_train_val_test('metr-la','metr-la_speed.npz',12)

train x:  (13700, 12, 207, 4) y: (13700, 12, 207, 4)
val x:  (0, 12, 207, 4) y: (0, 12, 207, 4)
test x:  (3425, 12, 207, 4) y: (3425, 12, 207, 4)
